### Bloc Type 1 — Définir les chemins et paramètres Go/NoGo

Adaptez uniquement ,  ou les fenêtres temporelles si vos fichiers sont ailleurs ou si vous souhaitez explorer d'autres composantes.


In [ ]:


RANDOM_SEED = 42
WITHIN_SUBJECT_SPLITS = 5

CLASSIFIER_CONFIG = {
    "LogisticRegression": {"type": "logreg", "params": {"penalty":"l1", "class_weight": "balanced", "max_iter": 500, "solver": "lbfgs"}},
    "LinearSVM": {"type": "linear_svm", "params": {"class_weight": "balanced", "max_iter": 5000}},
    "LDA": {"type": "lda", "params": {}},
}

FEATURES_OUT = RESULTS_DIR / "features_gonogo_ml.csv"
RESULTS_SUMMARY_OUT = RESULTS_DIR / "results_gonogo_ml.json"

QC_PLOTS = True
XDAWN_ENABLED = False
RUN_PERMUTATION_TEST = False
PERMUTATION_MODEL = "LogisticRegression"
PERMUTATION_N = 200

MNE_LOG_LEVEL = "WARNING"
PRINT_PROGRESS = True


In [ ]:


def _safe_cycles_per_freq(
    freqs: np.ndarray,
    epoch_len: float,
    min_cycles: float,
    max_cycles: float,
    safe_fraction: float
) -> np.ndarray:
    """
    Compute per-frequency n_cycles so each Morlet's temporal support
    (approx ~ n_cycles / f) stays <= safe_fraction * epoch_len.
    """
    # Max cycles allowed so that (n_cycles / f) <= safe_fraction * epoch_len
    max_allowed = safe_fraction * epoch_len * freqs
    n_cycles = np.minimum(max_cycles, np.maximum(min_cycles, max_allowed))
    return n_cycles

def compute_tfr_band_power(
    epochs: Epochs,
    picks: Sequence[str],
    freq_range: Tuple[float, float],
    n_freqs: int,
    tmin: float,
    tmax: float,
    baseline: Optional[Tuple[float, float]],
    mode: str,
    n_jobs: Optional[int],
    # New optional controls (pulled from TFR_CONFIG):
    method: Optional[str] = None,
    min_cycles: Optional[float] = None,
    max_cycles: Optional[float] = None,
    safe_epoch_fraction: Optional[float] = None,
    hilbert_fallback: bool = True,
) -> np.ndarray:
    if not picks:
        return np.full(len(epochs), np.nan)

    method = method or "morlet"
    epoch_len = float(epochs.times[-1] - epochs.times[0])

    fmin, fmax = freq_range
    freqs = np.linspace(fmin, fmax, n_freqs)

    if method.lower() == "hilbert":
        return compute_band_envelope_hilbert(
            epochs, picks, fmin, fmax, tmin, tmax, n_jobs=n_jobs
        )

    # Morlet path with safety constraints
    min_cycles = 2.0 if min_cycles is None else float(min_cycles)
    max_cycles = 7.0 if max_cycles is None else float(max_cycles)
    safe_epoch_fraction = 0.50 if safe_epoch_fraction is None else float(safe_epoch_fraction)

    # cycles per freq so that (n_cycles / f) <= safe_fraction * epoch_len
    n_cycles = _safe_cycles_per_freq(freqs, epoch_len, min_cycles, max_cycles, safe_epoch_fraction)

    # If still unsafe for very low f, use Hilbert fallback
    # (e.g., epochs are very short or baseline eats too much)
    longest_support = np.max(n_cycles / freqs)  # seconds
    if hilbert_fallback and (longest_support > safe_epoch_fraction * epoch_len + 1e-9):
        return compute_band_envelope_hilbert(
            epochs, picks, fmin, fmax, tmin, tmax, n_jobs=n_jobs
        )

    power = tfr_morlet(
        epochs, freqs=freqs, n_cycles=n_cycles, picks=picks,
        average=False, return_itc=False, use_fft=True,
        n_jobs=n_jobs, verbose="ERROR"
    )
    if baseline is not None:
        power.apply_baseline(baseline=baseline, mode=mode)
    mask = _window_mask(power.times, tmin, tmax)
    if not mask.any():
        return np.full(len(epochs), np.nan)
    data = power.data[:, :, :, mask]  # (n_trials, n_ch, n_freq, n_time)
    return data.mean(axis=(1, 2, 3))


def build_feature_table(epochs: Epochs, labels: np.ndarray, subject_id: str, source_path: Path) -> pd.DataFrame:
    """Assemble single-trial ERP and TFR features for one recording."""
    picks_cache: Dict[str, List[str]] = {}

    def cluster_picks(name: str) -> List[str]:
        if name not in CHANNEL_CLUSTERS:
            raise KeyError(f"Unknown cluster '{name}'.")
        if name not in picks_cache:
            picks_cache[name] = get_cluster_channels(epochs, CHANNEL_CLUSTERS[name])
        return picks_cache[name]

    feature_map: Dict[str, np.ndarray] = {}
    for spec in ERP_MEAN_WINDOWS:
        picks = cluster_picks(spec["cluster"])
        feature_map[spec["name"]] = mean_amplitude(epochs, picks, spec["tmin"], spec["tmax"])
    for spec in ERP_PEAK_WINDOWS:
        picks = cluster_picks(spec["cluster"])
        values, latencies = peak_value_and_latency(epochs, picks, spec["tmin"], spec["tmax"], spec["mode"])
        feature_map[f"{spec['name']}_value"] = values
        feature_map[f"{spec['name']}_latency"] = latencies
    if TFR_CONFIG.get("enabled", False):
        for spec in TFR_CONFIG.get("specs", []):
            picks = cluster_picks(spec["cluster"])
            feature_map[spec["name"]] = compute_tfr_band_power(
                epochs=epochs,
                picks=picks,
                freq_range=spec["freq_range"],
                n_freqs=spec["n_freqs"],
                tmin=spec["tmin"],
                tmax=spec["tmax"],
                baseline=TFR_CONFIG.get("baseline"),
                mode=TFR_CONFIG.get("mode", "logratio"),
                n_jobs=TFR_CONFIG.get("n_jobs"),
                method=spec.get("method", TFR_CONFIG.get("method", "morlet")),
                min_cycles=TFR_CONFIG.get("min_cycles", 2.0),
                max_cycles=TFR_CONFIG.get("max_cycles", 7.0),
                safe_epoch_fraction=TFR_CONFIG.get("safe_epoch_fraction", 0.50),
                hilbert_fallback=TFR_CONFIG.get("hilbert_fallback", True),
            )

    base_info = {
        "subject_id": subject_id,
        "source_file": source_path.name,
        "source_path": str(source_path),
        "trial_index": np.arange(len(labels)),
        "condition": labels,
    }
    return pd.DataFrame({**base_info, **feature_map})


def instantiate_classifier(name: str, spec: Dict[str, object]):
    """Build a StandardScaler + classifier pipeline from the config entry."""
    kind = spec.get("type")
    params = dict(spec.get("params", {}))
    if kind == "logreg":
        params.setdefault("random_state", RANDOM_SEED)
        estimator = LogisticRegression(**params)
    elif kind == "linear_svm":
        params.setdefault("random_state", RANDOM_SEED)
        estimator = LinearSVC(**params)
    elif kind == "lda":
        estimator = LinearDiscriminantAnalysis(**params)
    else:
        raise ValueError(f"Unknown classifier type '{kind}'.")
    return make_pipeline(StandardScaler(), estimator)


def evaluate_within_subject(
    X: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    classifier_config: Dict[str, Dict[str, object]],
    n_splits: int,
) -> Dict[str, Dict[str, object]]:
    """Compute stratified K-fold balanced accuracy per subject for each model."""
    results: Dict[str, Dict[str, object]] = {}
    unique_subjects = np.unique(subjects)
    for name, spec in classifier_config.items():
        subject_scores: Dict[str, float] = {}
        for subject in unique_subjects:
            mask = subjects == subject
            if mask.sum() < 2:
                continue
            y_sub = y[mask]
            if np.unique(y_sub).size < 2:
                continue
            class_counts = np.bincount(y_sub)
            class_counts = class_counts[class_counts > 0]
            if class_counts.size == 0:
                continue
            splits = min(n_splits, int(class_counts.min()))
            if splits < 2:
                continue
            cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_SEED)
            pipeline = instantiate_classifier(name, spec)
            scores = cross_val_score(pipeline, X[mask], y_sub, cv=cv, scoring="balanced_accuracy")
            subject_scores[subject] = float(np.mean(scores))
        results[name] = {
            "per_subject": subject_scores,
            "mean_balanced_accuracy": float(np.mean(list(subject_scores.values())))
            if subject_scores
            else float("nan"),
        }
    return results


def _continuous_scores(model, X_test: np.ndarray) -> Optional[np.ndarray]:
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_test)
    elif hasattr(model, "predict_proba"):
        scores = model.predict_proba(X_test)[:, 1]
    else:
        return None
    if isinstance(scores, (list, tuple)):
        scores = np.asarray(scores)
    if scores.ndim > 1:
        scores = scores[:, 0]
    return scores


def safe_roc_auc(y_true: np.ndarray, scores: Optional[np.ndarray]) -> float:
    if scores is None or np.unique(y_true).size < 2:
        return float("nan")
    try:
        return float(roc_auc_score(y_true, scores))
    except ValueError:
        return float("nan")


def evaluate_loso(
    X: np.ndarray,
    y: np.ndarray,
    subjects: np.ndarray,
    classifier_config: Dict[str, Dict[str, object]],
) -> Dict[str, Dict[str, object]]:
    """Perform leave-one-subject-out evaluation for each classifier."""
    results: Dict[str, Dict[str, object]] = {}
    logo = LeaveOneGroupOut()
    for name, spec in classifier_config.items():
        folds = []
        for train_idx, test_idx in logo.split(X, y, groups=subjects):
            if np.unique(y[train_idx]).size < 2 or np.unique(y[test_idx]).size < 2:
                continue
            pipeline = instantiate_classifier(name, spec)
            pipeline.fit(X[train_idx], y[train_idx])
            y_pred = pipeline.predict(X[test_idx])
            y_score = _continuous_scores(pipeline, X[test_idx])
            folds.append(
                {
                    "held_out_subject": str(subjects[test_idx][0]),
                    "balanced_accuracy": float(balanced_accuracy_score(y[test_idx], y_pred)),
                    "f1": float(f1_score(y[test_idx], y_pred)),
                    "roc_auc": safe_roc_auc(y[test_idx], y_score),
                    "n_test": int(len(test_idx)),
                }
            )
        if folds:
            roc_values = [fold["roc_auc"] for fold in folds if not math.isnan(fold["roc_auc"])]
            results[name] = {
                "folds": folds,
                "mean_balanced_accuracy": float(np.mean([fold["balanced_accuracy"] for fold in folds])),
                "mean_f1": float(np.mean([fold["f1"] for fold in folds])),
                "mean_roc_auc": float(np.mean(roc_values)) if roc_values else float("nan"),
            }
        else:
            results[name] = {
                "folds": [],
                "mean_balanced_accuracy": float("nan"),
                "mean_f1": float("nan"),
                "mean_roc_auc": float("nan"),
            }
    return results


## 4. Construire et comparer des modèles linéaires

Nous passons des features au machine learning : vecteur , labels  (0=Go, 1=NoGo) et groupes par sujet.


### Bloc Type 2 — Configurer les matrices et lancer les évaluations


In [39]:

if 'FEATURES_DF' not in globals():
    raise RuntimeError("Exécutez d'abord l'extraction des features.")

feature_df = FEATURES_DF.copy()
non_feature_cols = {"subject_id", "source_file", "source_path", "trial_index", "condition"}
numeric_cols = [
    col
    for col in feature_df.columns
    if col not in non_feature_cols and pd.api.types.is_numeric_dtype(feature_df[col])
]
if not numeric_cols:
    raise RuntimeError("Pas de colonne numérique disponible pour le ML.")

X = feature_df[numeric_cols].to_numpy(dtype=float)
valid_mask = np.isfinite(X).all(axis=1)
if not np.all(valid_mask):
    removed = int((~valid_mask).sum())
    print(f"[info] {removed} essais retirés (NaN/inf).")
    feature_df = feature_df.loc[valid_mask].reset_index(drop=True)
    X = X[valid_mask]

y = (feature_df["condition"] == "NoGo").astype(int).to_numpy()
groups = feature_df["subject_id"].to_numpy()

within_subject_results = evaluate_within_subject(X, y, groups, CLASSIFIER_CONFIG, WITHIN_SUBJECT_SPLITS)
loso_results = evaluate_loso(X, y, groups, CLASSIFIER_CONFIG)

rows = []
for name in CLASSIFIER_CONFIG.keys():
    rows.append(
        {
            "model": name,
            "within_subject_bal_acc": within_subject_results.get(name, {}).get("mean_balanced_accuracy", float("nan")),
            "loso_bal_acc": loso_results.get(name, {}).get("mean_balanced_accuracy", float("nan")),
            "loso_f1": loso_results.get(name, {}).get("mean_f1", float("nan")),
            "loso_roc_auc": loso_results.get(name, {}).get("mean_roc_auc", float("nan")),
        }
    )
display(pd.DataFrame(rows))

results_summary = {
    "n_subjects": int(np.unique(groups).size),
    "n_trials": int(len(y)),
    "feature_columns": numeric_cols,
    "within_subject": within_subject_results,
    "loso": loso_results,
}
with open(RESULTS_SUMMARY_OUT, "w", encoding="utf-8") as handle:
    json.dump(results_summary, handle, indent=2)
print(f"Résumé enregistré dans {RESULTS_SUMMARY_OUT.resolve()}")


,model,within_subject_bal_acc,loso_bal_acc,loso_f1,loso_roc_auc
0,LogisticRegression,0.615,0.604,0.400,0.649
1,LinearSVM,0.606,0.609,0.406,0.647
2,LDA,0.581,0.521,0.116,0.649


Résumé enregistré dans D:\Yann\neurotheque_resources\derivatives\results_gonogo_ml.json


### Bloc Type 3 — Test de permutation (optionnel)


In [40]:

if "X" not in globals() or "y" not in globals():
    raise RuntimeError("Lancez d'abord l'évaluation principale (matrices X/y).")
if PERMUTATION_MODEL not in CLASSIFIER_CONFIG:
    raise ValueError(f"{PERMUTATION_MODEL} n'est pas défini dans CLASSIFIER_CONFIG.")
estimator = instantiate_classifier(PERMUTATION_MODEL, CLASSIFIER_CONFIG[PERMUTATION_MODEL])
class_counts = np.bincount(y)
class_counts = class_counts[class_counts > 0]
if class_counts.size == 0:
    raise RuntimeError("Labels vides : permutation impossible.")
splits = min(WITHIN_SUBJECT_SPLITS, int(class_counts.min()))
if splits < 2:
    print("Permutation ignorée (trop peu d'essais par classe).")
else:
    cv = StratifiedKFold(n_splits=splits, shuffle=True, random_state=RANDOM_SEED)
    score, perm_scores, pvalue = permutation_test_score(
        estimator,
        X,
        y,
        cv=cv,
        n_permutations=PERMUTATION_N,
        scoring="balanced_accuracy",
        random_state=RANDOM_SEED,
    )
    print(f"Score observé : {score:.3f}")
    print(f"p-value ({PERMUTATION_N} permutations) : {pvalue:.3f}")



Score observé : 0.630
p-value (200 permutations) : 0.005
